In [1]:
import numpy as np
from typing import Tuple, Dict
from sklearn.tree import DecisionTreeRegressor
import matplotlib.pyplot as plt
import pandas as pd
import os
from utils import *

new_data = np.linspace(0, 1, 4) 
seed = 42
dt_args = {"max_leaf_nodes": 5}

####### Simulation parameters  
n_sim = 1000
n = 6
B = 3          

In [4]:
# data and weights for X
rng = np.random.default_rng(seed)
boot_weights = np.zeros(n)
boot_weights += 1 / int(n/2)
index_drop = rng.choice(range(n), size=int(n/2), replace=False)
boot_weights[index_drop] = 0.0





# Seed adjustment
adjusted_seed = seed + 0
rng = np.random.default_rng(adjusted_seed)

# Generate data
x_train = rng.uniform(0, 1, n)
y_train = generate_response(x_train = x_train, seed=adjusted_seed)


# Perform bagging
tree_predictions_b, N_bi = bagging_decision_trees(
    x_train=x_train,
    y_train=y_train,
    new_data=new_data,
    B=B,
    dt_args=dt_args,
    seed=adjusted_seed,
    boot_weights=boot_weights,
)
bl_predictions = tree_predictions_b.mean(axis=0)


biased_var_estimate, bias_correction = ij_w_variance(N_bi=N_bi, T_N_b=tree_predictions_b, boot_weights=boot_weights)

bias_correction

c:\Users\rehan\meine_Repos\Masterarbeit\Paper\motivation_example2\Figure_2.11\utils.py:109: RuntimeWarning: invalid value encountered in divide
  array = cov_i_hoch2/((boot_weights.reshape(-1,1))**2)


array([0., 0., 0., 0.])

In [ ]:
################################################################################### Funktion noch überprüfen
def ij_w_variance( N_bi: np.ndarray, T_N_b: np.ndarray, boot_weights: np.ndarray):

    B, n = N_bi.shape
    T_N_b_mean = np.mean(T_N_b, axis=0)
    n_plus = np.sum(boot_weights > 0)

    cov_i = ((N_bi - n * boot_weights.reshape(1,-1)).T @ (T_N_b - T_N_b_mean)) / B
    cov_i_hoch2 = cov_i**2
    array = cov_i_hoch2/((boot_weights.reshape(-1,1))**2)

    biased_var_estimate = np.sum(array[~np.isnan(array) & ~np.isinf(array)], axis=0) * (1/n_plus**2)


    biased_var_estimate = np.sum(cov_i_hoch2, axis=0)

    bias_correction =  (1/n_plus**2)  * np.var(T_N_b, axis=0, ddof=1)* n / B * np.sum( ( 1 / (boot_weights[boot_weights > 0] ) ) -1) 

    return biased_var_estimate, bias_correction